# 00 – Project Setup

**Phase 1** of the Philippine Crop Production and Climate Risk Lakehouse.
 
This notebook:
1. Creates the Unity Catalog catalog and schemas
2. Creates the `raw` volume under `source_data`
3. Confirms source CSV files are present in the volume

## 1. Configuration

In [0]:
from configs.project_config import (
    CATALOG,
    SCHEMAS,
    VOLUME_SCHEMA,
    VOLUME_NAME,
    VOLUME_PATH,
    PATHS,
    BRONZE_TABLES,
    SILVER_TABLES,
    GOLD_TABLES,
    ML_TABLES,
    MODEL_NAME
)

print(f'Catalog      : {CATALOG}')
print(f'Volume path  : {VOLUME_PATH}')
print(f'Crop CSV     : {PATHS.crop_csv}')
print(f'Weather CSV  : {PATHS.weather_csv}')

## 2. Create Catalog & Schemas

In [0]:
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'USE CATALOG {CATALOG}')

for schema_name in SCHEMAS.values():
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}')
    print(f'Schema ready: {CATALOG}.{schema_name}')

## 3. Create Volume for source CSVs

In [0]:
spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.{VOLUME_SCHEMA}.{VOLUME_NAME}')

print(f'Volume Ready: {VOLUME_PATH}')

## 4. Verify source files in the volume

Upload these two files to `/Volumes/crop_medallion_architecture/source_data/raw`:
- `crop_trend.csv`
- `province_weather_quarterly.csv`

In [0]:
def check_file(path: str) -> dict:
    try:
        df = spark.read.option("header", True).option("inferSchema", False).csv(path)
        return {"path": path, "exists": True, "rows": df.count(), "cols": len(df.columns)}
    except Exception as e:
        return {"path": path, "exists": False, "error": str(e)[:150]}

results = [check_file(PATHS.crop_csv), check_file(PATHS.weather_csv)]

for r in results:
    print("-" * 50)
    print(f"  path   : {r['path']}")
    print(f"  exists : {r['exists']}")
    if r["exists"]:
        print(f"  rows   : {r['rows']}")
        print(f"  cols   : {r['cols']}")
    else:
        print(f"  error  : {r['error']}")
